# 11 — Final Forecasts and Thesis Visualizations

This notebook creates presentation-ready tables and figures from approved predictions and
Notebook 10 comparison outputs. It does not retrain, refit, or select model parameters.


## 1. Imports, frozen inputs, and output folders


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", context="notebook", palette="colorblind",
              rc={"figure.dpi": 120, "savefig.dpi": 300})

def locate_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "results" / "hybrid" / "predictions.csv").exists():
            return candidate
    raise FileNotFoundError("Run from the repository root or notebooks directory.")

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def markdown_table(frame: pd.DataFrame) -> str:
    headers = [str(column) for column in frame.columns]
    lines = ["| " + " | ".join(headers) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in frame.itertuples(index=False, name=None):
        lines.append("| " + " | ".join(str(value) for value in row) + " |")
    return "\n".join(lines)

PROJECT_ROOT = locate_project_root(Path.cwd())
FINAL_RESULTS = PROJECT_ROOT / "results" / "final"
FINAL_FIGURES = PROJECT_ROOT / "figures" / "final"
for directory in (FINAL_RESULTS, FINAL_FIGURES):
    directory.mkdir(parents=True, exist_ok=True)

THESIS_RESULTS = PROJECT_ROOT / "results" / "thesis_tables"
THESIS_FIGURES = PROJECT_ROOT / "figures" / "thesis"
for directory in (THESIS_RESULTS, THESIS_FIGURES):
    directory.mkdir(parents=True, exist_ok=True)

comparison = pd.read_csv(FINAL_RESULTS / "model_comparison_table.csv")
ranking = pd.read_csv(FINAL_RESULTS / "final_model_ranking.csv")
regional_comparison = pd.read_csv(FINAL_RESULTS / "regional_performance_comparison.csv")


## 2. Select the presentation model from the approved ranking

The presentation model is the highest-ranked approach for which approved prediction rows exist.
This is not a claim of universal superiority. It is a reproducible presentation decision tied
to the multi-criterion ranking and the supplied 2022 test outputs.


In [ ]:
prediction_sources = {
    "ARIMA": PROJECT_ROOT / "results" / "arima" / "predictions.csv",
    "Random Forest": PROJECT_ROOT / "results" / "random_forest" / "predictions.csv",
    "XGBoost": PROJECT_ROOT / "results" / "xgboost" / "predictions.csv",
    "Hybrid": PROJECT_ROOT / "results" / "hybrid" / "predictions.csv",
}
eligible = ranking[ranking["model_name"].isin(prediction_sources)].sort_values("overall_rank")
if eligible.empty:
    raise ValueError("No ranked model has approved prediction rows.")
selected_model = eligible.iloc[0]["model_name"]
selected_path = prediction_sources[selected_model]
selected_predictions = pd.read_csv(selected_path)
if selected_model == "Hybrid":
    selected_predictions = selected_predictions.query(
        "model_name == 'ARIMA + Random Forest + XGBoost'"
    ).copy()
selected_predictions["model_name"] = selected_model

test_forecasts = selected_predictions.query("dataset_split == 'test'").copy()
test_forecasts = test_forecasts[[
    "region", "year", "actual", "predicted", "model_name", "residual",
    "absolute_error", "squared_error",
]].rename(columns={
    "actual": "actual_consumption_kwh",
    "predicted": "predicted_consumption_kwh",
    "model_name": "model_used",
    "residual": "error_kwh",
})
test_forecasts["absolute_percentage_error"] = (
    100 * test_forecasts["absolute_error"] / test_forecasts["actual_consumption_kwh"]
)
test_forecasts = test_forecasts.sort_values("region").reset_index(drop=True)
display(test_forecasts)
print("Presentation model:", selected_model)


## 3. Publication-ready tables


In [ ]:
performance_table = comparison.query("dataset_split == 'test'").merge(
    ranking[["model_name", "overall_rank", "mean_numeric_rank",
             "complexity", "interpretability", "robustness_note"]],
    on="model_name", how="left"
).sort_values("overall_rank")
performance_table.to_csv(THESIS_RESULTS / "model_performance_table.csv",
                         index=False, float_format="%.15g")

regional_accuracy = test_forecasts[[
    "region", "year", "actual_consumption_kwh", "predicted_consumption_kwh",
    "absolute_error", "absolute_percentage_error", "model_used",
]].sort_values("absolute_percentage_error")
regional_accuracy.to_csv(
    THESIS_RESULTS / "regional_forecasting_accuracy_table.csv",
    index=False, float_format="%.15g"
)
test_forecasts.to_csv(
    THESIS_RESULTS / "final_forecast_table.csv", index=False, float_format="%.15g"
)
print("Saved three publication-ready tables.")


## 4. Thesis-ready figures


In [ ]:
# Observed 2021–2022 regional trajectories available in approved predictions
observed = selected_predictions[["region", "year", "actual"]].drop_duplicates().sort_values(
    ["region", "year"]
)
fig, ax = plt.subplots(figsize=(14, 8))
for region, frame in observed.groupby("region"):
    ax.plot(frame["year"], frame["actual"]/1e9, marker="o", label=region)
ax.set_xticks(sorted(observed["year"].unique())); ax.set_ylabel("Consumption (billion kWh)")
ax.set_title("Saudi Regional Household Consumption — Approved Holdout Years", weight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.tight_layout(); fig.savefig(THESIS_FIGURES / "saudi_regional_consumption_trends.png",
                                bbox_inches="tight"); plt.show()

ordered = test_forecasts.sort_values("actual_consumption_kwh")
fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(ordered["region"], ordered["actual_consumption_kwh"]/1e9,
        marker="o", linewidth=2.5, label="Observed")
ax.plot(ordered["region"], ordered["predicted_consumption_kwh"]/1e9,
        marker="s", linewidth=2.2, label=selected_model)
ax.tick_params(axis="x", rotation=70); ax.set_ylabel("Consumption (billion kWh)")
ax.set_title("Observed versus Predicted Regional Consumption — 2022", weight="bold")
ax.legend(frameon=False); fig.tight_layout()
fig.savefig(THESIS_FIGURES / "observed_vs_predicted_consumption.png", bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(8, 7))
maximum = max(ordered["actual_consumption_kwh"].max(),
              ordered["predicted_consumption_kwh"].max()) / 1e9
ax.scatter(ordered["actual_consumption_kwh"]/1e9,
           ordered["predicted_consumption_kwh"]/1e9, s=70)
ax.plot([0, maximum], [0, maximum], linestyle="--", color="black")
for row in ordered.itertuples():
    ax.annotate(row.region, (row.actual_consumption_kwh/1e9,
                            row.predicted_consumption_kwh/1e9), fontsize=8)
ax.set_xlabel("Actual (billion kWh)"); ax.set_ylabel("Predicted (billion kWh)")
ax.set_title(f"{selected_model}: 2022 Regional Forecasts", weight="bold")
fig.tight_layout(); fig.savefig(THESIS_FIGURES / "best_model_forecast_visualization.png",
                                bbox_inches="tight"); plt.show()

advanced_test = regional_comparison.query("dataset_split == 'test'")
fig, ax = plt.subplots(figsize=(14, 7))
sns.barplot(data=advanced_test, x="region", y=advanced_test["MAE"]/1e9,
            hue="model_name", ax=ax)
ax.tick_params(axis="x", rotation=70); ax.set_ylabel("Absolute error (billion kWh)")
ax.set_title("Regional Forecast Comparison", weight="bold")
fig.tight_layout(); fig.savefig(THESIS_FIGURES / "regional_forecast_comparison.png",
                                bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.barplot(data=ordered, x=ordered["absolute_error"]/1e9, y="region",
            color="#C8553D", ax=axes[0])
axes[0].set_title("Absolute Error"); axes[0].set_xlabel("Billion kWh"); axes[0].set_ylabel("")
sns.scatterplot(data=ordered, x="predicted_consumption_kwh", y="error_kwh", ax=axes[1])
axes[1].axhline(0, color="black", linewidth=1); axes[1].set_title("Residual Pattern")
fig.suptitle(f"{selected_model} Error Analysis", fontsize=17, weight="bold")
fig.tight_layout(); fig.savefig(THESIS_FIGURES / "error_analysis.png", bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
plot = performance_table.sort_values("RMSE")
sns.barplot(data=plot, x=plot["RMSE"]/1e9, y="model_name", color="#1F4E78", ax=ax)
ax.set_xlabel("2022 test RMSE (billion kWh)"); ax.set_ylabel("")
ax.set_title("Model Performance Summary", weight="bold")
fig.tight_layout(); fig.savefig(THESIS_FIGURES / "model_performance_summary.png",
                                bbox_inches="tight"); plt.show()


## 5. Final interpretation


In [ ]:
selected_row = ranking.query("model_name == @selected_model").iloc[0]
conclusion = f'''# Final Forecast Presentation Note

The presentation tables use **{selected_model}**, the highest-ranked approach with approved
prediction rows (overall numerical rank {int(selected_row["overall_rank"])}). This is a
conditional presentation choice, not evidence that the model is universally superior.

The conclusion is limited by four model-ready target years, 13 heterogeneous regions, only one
final test year, and uncertainty in every fitted model. Improvements should be considered
alongside simpler baselines and interpretability. Feature importance and predictive accuracy
must not be interpreted causally.
'''
(THESIS_RESULTS / "final_interpretation.md").write_text(conclusion, encoding="utf-8")
print(conclusion)


## 6. Phase 5 validation summary and output manifest


In [ ]:
validation_summary = f'''# Phase 5 Validation Summary

Generated: {datetime.now().astimezone().isoformat(timespec="seconds")}

## Notebooks created

- `10_Model_Evaluation_and_Comparison.ipynb`
- `11_Final_Forecasts_and_Visualizations.ipynb`

## Frozen inputs used

- `results/baseline_results.csv`
- `results/arima/metrics.csv` and `predictions.csv`
- `results/random_forest/metrics.csv` and `predictions.csv`
- `results/xgboost/metrics.csv` and `predictions.csv`
- `results/hybrid/metrics.csv` and `predictions.csv`
- Notebook 10 approved comparison tables

## Outputs

- Five final evaluation/result files under `results/final/`
- Eight final comparison figures under `figures/final/`
- Four thesis table/note files under `results/thesis_tables/`
- Six thesis figures under `figures/thesis/`

## Reproducibility and integrity

- No model was retrained or rerun.
- No parameter, dataset, feature, split, or previous result was modified.
- Prediction keys and actual values were aligned before comparison.
- Baseline prediction-level analyses were not invented because Phase 3 saved metrics only.
- Formal significance tests were not performed because the sample is too small and spatially dependent.
- The official Phase 1–4 repository was hashed before Phase 5 and remains unchanged.

## Interpretation

The selected presentation model is `{selected_model}`, based on a multi-criterion numerical
ranking. The conclusion remains conditional on one 2022 test year and does not imply causation.
'''
summary_path = PROJECT_ROOT / "results" / "phase5_validation_summary.md"
summary_path.write_text(validation_summary, encoding="utf-8")

phase5_roots = [FINAL_RESULTS, FINAL_FIGURES, THESIS_RESULTS, THESIS_FIGURES]
rows = []
for base in phase5_roots:
    for path in sorted(base.rglob("*")):
        if not path.is_file():
            continue
        generator = (
            "10_Model_Evaluation_and_Comparison.ipynb"
            if base in [FINAL_RESULTS, FINAL_FIGURES]
            else "11_Final_Forecasts_and_Visualizations.ipynb"
        )
        row_count = column_count = ""
        if path.suffix.lower() == ".csv":
            table = pd.read_csv(path)
            row_count, column_count = len(table), table.shape[1]
        rows.append({
            "output_filename": str(path.relative_to(PROJECT_ROOT)),
            "generating_notebook": generator,
            "row_count": row_count,
            "column_count": column_count,
            "sha256": sha256(path),
        })
for path, generator in [
    (PROJECT_ROOT / "notebooks" / "10_Model_Evaluation_and_Comparison.ipynb",
     "10_Model_Evaluation_and_Comparison.ipynb"),
    (PROJECT_ROOT / "notebooks" / "11_Final_Forecasts_and_Visualizations.ipynb",
     "11_Final_Forecasts_and_Visualizations.ipynb"),
    (summary_path, "11_Final_Forecasts_and_Visualizations.ipynb"),
]:
    rows.append({
        "output_filename": str(path.relative_to(PROJECT_ROOT)),
        "generating_notebook": generator,
        "row_count": "",
        "column_count": "",
        "sha256": sha256(path),
    })
manifest = pd.DataFrame(rows).drop_duplicates("output_filename").sort_values("output_filename")
manifest.to_csv(PROJECT_ROOT / "results" / "phase5_output_manifest.csv", index=False)
print("Phase 5 manifest rows:", len(manifest))


## Conclusion

Phase 5 is complete. The project now contains final comparison outputs and thesis-ready
presentation artifacts, without changing or rerunning any earlier phase.
